# NASA PCoE Battery Degradation EDA

This notebook builds the chronological discharge table, audits capacity quality, derives SOH against the documented 2.0 Ah rating, and assigns RUL only to batteries with an observable above-threshold-to-EOL trajectory.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'PROJECT_ROADMAP.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.battery import (
    build_battery_discharge_table,
    load_battery_metadata,
    summarize_battery_discharge_table,
)

FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'battery'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'battery'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

## Build and save the chronological discharge table

In [ ]:
metadata = load_battery_metadata()
discharge = build_battery_discharge_table(metadata)
processed_columns = [
    'battery_id', 'uid', 'filename', 'start_time', 'ambient_temperature',
    'test_id', 'discharge_cycle', 'Capacity', 'valid_capacity',
    'soh_percent', 'eol_capacity_ah', 'initial_valid_capacity_ah',
    'observed_eol_cycle', 'rul_cycles', 'at_or_after_documented_eol',
    'rul_label_status', 'rul_training_eligible',
]
discharge[processed_columns].to_csv(
    PROCESSED_DIR / 'discharge_cycles.csv', index=False
)
pd.Series(summarize_battery_discharge_table(discharge), name='value').to_frame()

## Capacity-label quality by battery

In [ ]:
quality = discharge.groupby('battery_id').agg(
    discharge_rows=('uid', 'size'),
    valid_capacity_rows=('valid_capacity', 'sum'),
    missing_capacity_rows=('Capacity', lambda values: values.isna().sum()),
    zero_capacity_rows=('Capacity', lambda values: (values == 0).sum()),
    initial_capacity_ah=('initial_valid_capacity_ah', 'first'),
    minimum_valid_capacity_ah=('Capacity', lambda values: values[values > 0].min()),
    maximum_valid_capacity_ah=('Capacity', 'max'),
    label_status=('rul_label_status', 'first'),
).sort_index()
quality

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(quality))
ax.bar(positions, quality['valid_capacity_rows'], label='Valid capacity', color='#0891b2')
ax.bar(positions, quality['missing_capacity_rows'] + quality['zero_capacity_rows'], bottom=quality['valid_capacity_rows'], label='Missing or zero', color='#ef4444')
ax.set_xticks(positions, quality.index, rotation=75)
ax.set(title='Battery Discharge Capacity-Label Quality', xlabel='Battery', ylabel='Discharge records')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'battery_capacity_label_quality.png', dpi=160)
plt.show()

## Capacity degradation trajectories

Different temperatures, load currents, and voltage cutoffs make cross-battery capacity levels non-identical. Trajectories are therefore shown separately and are not treated as one homogeneous experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
for battery_id, group in discharge.loc[discharge['valid_capacity']].groupby('battery_id'):
    ax.plot(group['discharge_cycle'], group['Capacity'], linewidth=1.2, alpha=0.75, label=battery_id)
ax.axhline(1.4, color='#ef4444', linestyle='--', linewidth=1, label='1.4 Ah documented EOL')
ax.axhline(1.6, color='#f59e0b', linestyle='--', linewidth=1, label='1.6 Ah documented EOL')
ax.set(title='Valid Discharge Capacity Trajectories', xlabel='Chronological discharge cycle', ylabel='Capacity (Ah)')
ax.legend(ncol=4, fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'battery_capacity_trajectories.png', dpi=160, bbox_inches='tight')
plt.show()

## SOH distribution and EOL-label availability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(discharge['soh_percent'].dropna(), bins=40, color='#7c3aed', edgecolor='white')
axes[0].axvline(70, color='#ef4444', linestyle='--', label='70% SOH')
axes[0].axvline(80, color='#f59e0b', linestyle='--', label='80% SOH')
axes[0].set(title='Observed SOH Distribution', xlabel='SOH (%)', ylabel='Records')
axes[0].legend()
status_counts = discharge.groupby('battery_id')['rul_label_status'].first().value_counts()
axes[1].bar(status_counts.index, status_counts.values, color=['#0891b2', '#64748b', '#f59e0b', '#ef4444'])
axes[1].set(title='RUL Label Status by Battery', xlabel='Status', ylabel='Batteries')
axes[1].tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'battery_soh_and_rul_status.png', dpi=160)
plt.show()

## Observed EOL trajectories

Only batteries that begin above their documented capacity threshold and later cross it receive direct RUL labels.

In [ ]:
observed = discharge.loc[discharge['rul_label_status'] == 'observed_eol']
eol_cycles = observed.groupby('battery_id')['observed_eol_cycle'].first().sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(eol_cycles.index, eol_cycles.values, color='#0ea5e9')
ax.set(title='First Observed Documented-EOL Cycle', xlabel='Battery', ylabel='Discharge cycle')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'battery_observed_eol_cycles.png', dpi=160)
plt.show()
eol_cycles.to_frame('observed_eol_cycle')

## Modeling implications

- SOH regression can use 2,750 positive-capacity discharge rows, but splitting must be grouped by battery ID.
- Direct supervised RUL labels are available for only nine complete trajectories, so uncertainty will be high.
- Twelve threshold-documented batteries begin at or below threshold and are left-censored.
- B0007 is right-censored because its documented threshold is not observed.
- Twelve batteries have no documented threshold, including the software-failure group B0049-B0052.
- Low-capacity anomalies and protocol differences require features for temperature, load, voltage cutoff, and battery identity/group.
- The final recorded discharge must never be treated automatically as EOL.